# Week 4 — First API Test
**Goal:** Confirm the Seoul Real-time City Data API works and understand its structure.

Questions we want to answer:
1. Does the API respond?
2. Is the response XML or JSON?
3. What are the top-level fields?
4. Where is the congestion/population information?
5. What field contains time?
6. What field contains location name?
7. What field contains congestion status?

## Step 1 — Raw request (paste your real key below)

In [6]:
import requests

API_KEY = "68646b6c69556c6b38366468747543"   # <-- replace this with your key
location = "광화문·덕수궁"

url = f"http://openapi.seoul.go.kr:8088/{API_KEY}/xml/citydata/1/5/{location}"

response = requests.get(url, timeout=30)

print("Status code:", response.status_code)
print("\n--- First 2000 characters of response ---")
print(response.text[:2000])

Status code: 200

--- First 2000 characters of response ---
<SeoulRtd.citydata><list_total_count>1</list_total_count><RESULT><RESULT.CODE>INFO-000</RESULT.CODE><RESULT.MESSAGE>정상 처리되었습니다.</RESULT.MESSAGE></RESULT><CITYDATA><AREA_NM>광화문·덕수궁</AREA_NM><AREA_CD>POI009</AREA_CD><LIVE_PPLTN_STTS><LIVE_PPLTN_STTS><AREA_NM>광화문·덕수궁</AREA_NM><AREA_CD>POI009</AREA_CD><AREA_CONGEST_LVL>여유</AREA_CONGEST_LVL><AREA_CONGEST_MSG>사람이 몰려있을 가능성이 낮고 붐빔은 거의 느껴지지 않아요. 도보 이동이 자유로워요.</AREA_CONGEST_MSG><AREA_PPLTN_MIN>6500</AREA_PPLTN_MIN><AREA_PPLTN_MAX>7000</AREA_PPLTN_MAX><MALE_PPLTN_RATE>58.1</MALE_PPLTN_RATE><FEMALE_PPLTN_RATE>41.9</FEMALE_PPLTN_RATE><PPLTN_RATE_0>0.6</PPLTN_RATE_0><PPLTN_RATE_10>3.7</PPLTN_RATE_10><PPLTN_RATE_20>24.3</PPLTN_RATE_20><PPLTN_RATE_30>22.5</PPLTN_RATE_30><PPLTN_RATE_40>19.1</PPLTN_RATE_40><PPLTN_RATE_50>16.5</PPLTN_RATE_50><PPLTN_RATE_60>8.2</PPLTN_RATE_60><PPLTN_RATE_70>5.1</PPLTN_RATE_70><RESNT_PPLTN_RATE>33.2</RESNT_PPLTN_RATE><NON_RESNT_PPLTN_RATE>66.8</NON_RESNT_PPLTN_RAT

## Step 2 — Parse XML and print all field names

In [7]:
import xml.etree.ElementTree as ET

root = ET.fromstring(response.text)

print("Root tag:", root.tag)
print("\n--- All tags found in response ---")

tags_seen = set()
for elem in root.iter():
    if elem.tag not in tags_seen:
        print(elem.tag)
        tags_seen.add(elem.tag)

Root tag: SeoulRtd.citydata

--- All tags found in response ---
SeoulRtd.citydata
list_total_count
RESULT
RESULT.CODE
RESULT.MESSAGE
CITYDATA
AREA_NM
AREA_CD
LIVE_PPLTN_STTS
AREA_CONGEST_LVL
AREA_CONGEST_MSG
AREA_PPLTN_MIN
AREA_PPLTN_MAX
MALE_PPLTN_RATE
FEMALE_PPLTN_RATE
PPLTN_RATE_0
PPLTN_RATE_10
PPLTN_RATE_20
PPLTN_RATE_30
PPLTN_RATE_40
PPLTN_RATE_50
PPLTN_RATE_60
PPLTN_RATE_70
RESNT_PPLTN_RATE
NON_RESNT_PPLTN_RATE
REPLACE_YN
PPLTN_TIME
FCST_YN
FCST_PPLTN
FCST_TIME
FCST_CONGEST_LVL
FCST_PPLTN_MIN
FCST_PPLTN_MAX
ROAD_TRAFFIC_STTS
AVG_ROAD_DATA
ROAD_MSG
ROAD_TRAFFIC_IDX
ROAD_TRAFFIC_SPD
ROAD_TRAFFIC_TIME
LINK_ID
ROAD_NM
START_ND_CD
START_ND_NM
START_ND_XY
END_ND_CD
END_ND_NM
END_ND_XY
DIST
SPD
IDX
XYLIST
PRK_STTS
PRK_NM
PRK_CD
PRK_TYPE
CPCTY
CUR_PRK_CNT
CUR_PRK_TIME
CUR_PRK_YN
PAY_YN
RATES
TIME_RATES
ADD_RATES
ADD_TIME_RATES
ADDRESS
ROAD_ADDR
LNG
LAT
SUB_STTS
SUB_STN_NM
SUB_STN_LINE
SUB_STN_RADDR
SUB_STN_JIBUN
SUB_STN_X
SUB_STN_Y
SUB_DETAIL
SUB_NT_STN
SUB_BF_STN
SUB_ROUTE_NM
SUB_LINE
S

## Step 3 — Find the key fields we need

In [8]:
# Fields we are looking for:
target_keywords = ["AREA", "TIME", "CONGEST", "PPLTN", "NAME", "STTS", "MIN", "MAX"]

print("--- Fields matching our keywords ---\n")
for elem in root.iter():
    for keyword in target_keywords:
        if keyword in elem.tag.upper() and elem.text:
            print(f"  {elem.tag}: {elem.text.strip()}")
            break

--- Fields matching our keywords ---

  AREA_NM: 광화문·덕수궁
  AREA_CD: POI009
  AREA_NM: 광화문·덕수궁
  AREA_CD: POI009
  AREA_CONGEST_LVL: 여유
  AREA_CONGEST_MSG: 사람이 몰려있을 가능성이 낮고 붐빔은 거의 느껴지지 않아요. 도보 이동이 자유로워요.
  AREA_PPLTN_MIN: 6500
  AREA_PPLTN_MAX: 7000
  MALE_PPLTN_RATE: 58.1
  FEMALE_PPLTN_RATE: 41.9
  PPLTN_RATE_0: 0.6
  PPLTN_RATE_10: 3.7
  PPLTN_RATE_20: 24.3
  PPLTN_RATE_30: 22.5
  PPLTN_RATE_40: 19.1
  PPLTN_RATE_50: 16.5
  PPLTN_RATE_60: 8.2
  PPLTN_RATE_70: 5.1
  RESNT_PPLTN_RATE: 33.2
  NON_RESNT_PPLTN_RATE: 66.8
  PPLTN_TIME: 2026-03-20 23:30
  FCST_TIME: 2026-03-21 00:00
  FCST_CONGEST_LVL: 여유
  FCST_PPLTN_MIN: 6000
  FCST_PPLTN_MAX: 6500
  FCST_TIME: 2026-03-21 01:00
  FCST_CONGEST_LVL: 여유
  FCST_PPLTN_MIN: 3500
  FCST_PPLTN_MAX: 4000
  FCST_TIME: 2026-03-21 02:00
  FCST_CONGEST_LVL: 여유
  FCST_PPLTN_MIN: 1500
  FCST_PPLTN_MAX: 2000
  FCST_TIME: 2026-03-21 03:00
  FCST_CONGEST_LVL: 여유
  FCST_PPLTN_MIN: 1000
  FCST_PPLTN_MAX: 1500
  FCST_TIME: 2026-03-21 04:00
  FCST_CONGEST_LVL:

## Step 4 — Extract the 5 most important fields cleanly

In [9]:
def find_tag(root, tag):
    """Find first element with this tag and return its text."""
    elem = root.find(f".//{tag}")
    return elem.text.strip() if elem is not None and elem.text else "NOT FOUND"

print("=== Key Field Summary ===")
print()

# Try the most common field names from Seoul API
# (these will print NOT FOUND if the name differs — check Step 2 output to fix)
print("Location name  :", find_tag(root, "AREA_NM"))
print("Observation time:", find_tag(root, "PPLTN_TIME"))
print("Congestion level:", find_tag(root, "AREA_CONGEST_LVL"))
print("Congestion msg  :", find_tag(root, "AREA_CONGEST_MSG"))
print("Population min  :", find_tag(root, "PPLTN_MIN"))
print("Population max  :", find_tag(root, "PPLTN_MAX"))

=== Key Field Summary ===

Location name  : 광화문·덕수궁
Observation time: 2026-03-20 23:30
Congestion level: 여유
Congestion msg  : 사람이 몰려있을 가능성이 낮고 붐빔은 거의 느껴지지 않아요. 도보 이동이 자유로워요.
Population min  : NOT FOUND
Population max  : NOT FOUND


## Step 5 — Save one raw response to disk (start collecting now!)

> Seoul does NOT provide historical data. Every call you save IS your dataset.
> This is one of the most important steps for your project.

In [10]:
import os
from datetime import datetime

# Create folder to store raw responses
os.makedirs("../data/raw", exist_ok=True)

# Save with timestamp in filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"../data/raw/gwanghwamun_{timestamp}.xml"

with open(filename, "w", encoding="utf-8") as f:
    f.write(response.text)

print(f"Saved to: {filename}")
print(f"File size: {os.path.getsize(filename)} bytes")

Saved to: ../data/raw/gwanghwamun_20260321_000025.xml
File size: 228232 bytes


## Step 6 — Summary: fill in after running all cells above

| Question | Answer |
|---|---|
| Does the API respond? | _(status code from Step 1)_ |
| Response format | XML |
| Location name field | `광화문·덕수궁` |
| Time field | `2026-03-20 23:25` |
| Congestion level field | `여유` |
| Population estimate fields | `6500`, `7000` |
| Historical data available? | **No — must collect yourself** |